# 1. Data Loading & Feature Engineering
**Indian Wind Dataset** | SiteID: 36565 | Lat: 23.03 N | Lon: 72.56 E | Year: 2014

- 8,760 hourly meteorological records
- Wind speed at 40m, 80m, 100m, 120m
- Wind direction, temperature, air pressure
- **Target**: Wind power computed using P = 0.5 * rho * A * Cp * v^3 (120m wind speed)


In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python: {sys.version}")
np.random.seed(42)
os.makedirs('results/predictions', exist_ok=True)
os.makedirs('results/plots', exist_ok=True)


## 1.1 Load Raw Dataset

In [ ]:
from data_pipeline import load_indian_dataset, engineer_features, prepare_data, compute_wind_power

CSV_PATH = r'36565_23.03_72.56_2014_cc6ea7f2b4966ba2f914d889439754cc.csv'

raw_df = load_indian_dataset(CSV_PATH)
print(f"Shape: {raw_df.shape}")
print(f"\nColumns: {list(raw_df.columns)}")
print(f"\nDate range: {raw_df['datetime'].min()} to {raw_df['datetime'].max()}")
raw_df.head(10)


## 1.2 Wind Power Computation

Using the simplified cubic power curve formula:

**P = 0.5 * rho * A * Cp * v^3**

| Parameter | Value |
|---|---|
| Air density (rho) | Dynamic (from P and T) |
| Rotor diameter | 28 m |
| Rotor area (A) | 615.75 m^2 |
| Power coefficient (Cp) | 0.40 |
| Cut-in speed | 3.0 m/s |
| Cut-out speed | 25.0 m/s |
| Rated power | 250 kW |


In [ ]:
# Plot the power curve
speeds = np.linspace(0, 30, 300)
power = compute_wind_power(speeds, np.full_like(speeds, 1.225))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(speeds, power, 'b-', linewidth=2.5)
ax.axvline(x=3.0, color='g', linestyle='--', alpha=0.7, label='Cut-in (3.0 m/s)')
ax.axvline(x=25.0, color='r', linestyle='--', alpha=0.7, label='Cut-out (25 m/s)')
ax.axhline(y=250, color='orange', linestyle='--', alpha=0.7, label='Rated (250 kW)')
ax.fill_between(speeds, power, alpha=0.15, color='blue')
ax.set_xlabel('Wind Speed (m/s)', fontsize=12)
ax.set_ylabel('Power (kW)', fontsize=12)
ax.set_title('Wind Turbine Power Curve: P = 0.5 * rho * A * Cp * v^3', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 1.3 Feature Engineering

15 features from raw meteorological data (plus derived air density):

| # | Feature | Description |
|---|---|---|
| 1-4 | wind_speed_40m/80m/100m/120m | Wind speed at 4 heights |
| 5-8 | wind_dir_40m/80m/100m/120m | Wind direction at 4 heights |
| 9-12 | temp_40m/80m/100m/120m | Temperature at 4 heights |
| 13-14 | pressure_40m/100m | Air pressure at 40m and 100m |
| 15 | air_density | Dynamic air density (rho) |


In [ ]:
feat_df = engineer_features(raw_df)
print(f"Engineered dataset shape: {feat_df.shape}")
print(f"\nColumns ({len(feat_df.columns)}):")
for i, col in enumerate(feat_df.columns):
    print(f"  {i+1:2d}. {col}")

print("\n--- Wind Power Statistics (kW) ---")
print(feat_df['wind_power'].describe())


## 1.4 Data Exploration

In [ ]:
# Time series plots
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

axes[0].plot(feat_df['datetime'], feat_df['wind_speed_120m'], color='#45B7D1', linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('Wind Speed (m/s)', fontsize=12)
axes[0].set_title('Wind Speed at 120m - Full Year 2014', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(feat_df['datetime'], feat_df['wind_power'], color='#E74C3C', linewidth=0.5, alpha=0.7)
axes[1].set_ylabel('Wind Power (Normalized)', fontsize=12)
axes[1].set_title('Computed Wind Power (P = 0.5 * rho * A * Cp * v^3) [Sabarmati 28m]', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(feat_df['datetime'], feat_df['temp_120m'], color='#FF6B6B', linewidth=0.5, alpha=0.7)
axes[2].set_ylabel('Temperature (C)', fontsize=12)
axes[2].set_title('Temperature at 120m', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/timeseries.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
corr = feat_df.drop(columns=['datetime']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, square=True, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(feat_df['wind_speed_120m'], bins=50, color='#45B7D1', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Wind Speed (m/s)')
axes[0].set_title('Wind Speed Distribution (120m)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].hist(feat_df['wind_power'], bins=50, color='#E74C3C', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Wind Power (Normalized)')
axes[1].set_title('Wind Power Distribution', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 1.5 Data Splitting & Normalization

In [ ]:
WINDOW_SIZE = 10
train_df, val_df, test_df, scaler, feature_cols, target_col = prepare_data(CSV_PATH, save_dir='results')

print(f"\nFeature columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols):
    print(f"  {i+1:2d}. {col}")
print(f"Target: {target_col}")


In [ ]:
print("=== Train (first 5 rows) ===")
print(train_df[feature_cols + [target_col]].head())
print("\n=== Validation (first 5 rows) ===")
print(val_df[feature_cols + [target_col]].head())
print("\n=== Test (first 5 rows) ===")
print(test_df[feature_cols + [target_col]].head())


## 1.6 Save Processed Data

In [ ]:
# Save processed dataframes
train_df.to_csv('results/train_data.csv', index=False)
val_df.to_csv('results/val_data.csv', index=False)
test_df.to_csv('results/test_data.csv', index=False)

# Save config
config = {
    'feature_cols': feature_cols,
    'target_col': target_col,
    'window_size': WINDOW_SIZE
}
with open('results/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("[OK] Saved files:")
print("  results/train_data.csv")
print("  results/val_data.csv")
print("  results/test_data.csv")
print("  results/config.json")
print("  results/scaler.save")
